# Data Preprocessing & Feature Engineering

This notebook covers text preprocessing and feature representations for genre identification and sentiment analysis:
- Text cleaning strategies for classification vs. sentiment analysis.
- Chorus extraction compared to full lyrics.
- Vectorization: Bag-of-Words, TF-IDF (unigrams & bigrams with sublinear TF), and LSA (TruncatedSVD).
- Feature selection using Chi-squared.
- Validation benchmark comparing preprocessing configurations.


## 1. Setup


In [2]:
# Install dependencies if not already present
!pip install scikit-learn seaborn matplotlib tqdm

In [3]:
import os
import sys
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack

# add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.pipeline import (
    clean_for_classification, 
    clean_for_sentiment, 
    extract_all_choruses
)

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, f1_score

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)


## 2. Dataset Loading
We load the training and test sets

In [4]:
possible_dirs = ['../dataset', 'dataset', '../datasets', 'datasets']
data_dir = next((d for d in possible_dirs if os.path.exists(d)), None)

if data_dir is None:
    raise FileNotFoundError("Could not find the dataset directory. Please verify dataset files.")

print(f"Loading datasets from: {os.path.abspath(data_dir)}")
train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))

print(f"Train set: {train_df.shape[0]:,} songs | Test set: {test_df.shape[0]:,} songs")

Loading datasets from: d:\Lean\season 5\Text Mining\Project\Text-Mining\dataset
Train set: 134,967 songs | Test set: 33,742 songs


## 3. Text Cleaning Strategies

We use two different cleaning setups depending on the downstream task:

1. **Genre Classification:**
   - Remove stopwords and punctuation to reduce noise and emphasize topical vocabulary.
   - Expand contractions (`n't` -> `not`, `'m` -> `am`, etc.).
   - Convert diacritics and lowercase.
   - Lemmatize verbs to collapse inflections (`running` -> `run`).

2. **Sentiment Analysis:**
   - Keep stopwords because words like `not`, `never`, and `very` carry sentiment and negation.
   - Keep punctuation because exclamation marks and question marks indicate intensity.
   - Keep original casing (capitalized words often convey emotion).
   - Skip lemmatization and stemming to preserve exact word forms.


In [ ]:
# clean lyrics for genre classification and sentiment analysis
print("Cleaning lyrics for Genre Identification (stopwords removed)...")
train_df['lyrics_clean_genre'] = train_df['lyrics'].apply(lambda x: clean_for_classification(x, max_words=1000))
test_df['lyrics_clean_genre'] = test_df['lyrics'].apply(lambda x: clean_for_classification(x, max_words=1000))

print("Cleaning lyrics for Sentiment Analysis (stopwords retained, contractions expanded)...")
train_df['lyrics_clean_sentiment'] = train_df['lyrics'].apply(lambda x: clean_for_sentiment(x, max_words=1000))
test_df['lyrics_clean_sentiment'] = test_df['lyrics'].apply(lambda x: clean_for_sentiment(x, max_words=1000))

# truncation check
raw_word_counts = train_df['lyrics'].astype(str).apply(lambda x: len(x.split()))
n_truncated = (raw_word_counts > 1000).sum()
pct_truncated = n_truncated / len(train_df) * 100
print(f"\nTruncation check: {n_truncated:,} songs ({pct_truncated:.1f}%) exceed 1,000 raw words and are capped.")
print(f"Median raw word count: {raw_word_counts.median():.0f} words — the 1,000-word cap retains full text for the vast majority of songs.")

print("\n--- Sample Comparison (First 100 characters) ---")
print("Raw:       ", train_df['lyrics'].iloc[0][:90].replace('\n', ' '))
print("Genre:     ", train_df['lyrics_clean_genre'].iloc[0][:90])
print("Sentiment: ", train_df['lyrics_clean_sentiment'].iloc[0][:90])


Cleaning lyrics for Genre Identification (stopwords removed)...
Cleaning lyrics for Sentiment Analysis (stopwords retained, contractions expanded)...


## 4. Chorus Extraction vs. Full Lyrics

The chorus often holds the central theme and hook of a song. However, explicit `[Chorus]` tags appear in only ~28.2% of songs, while the remaining 71.8% fall back to the first 6 lines. We test this extracted section against full lyrics to evaluate whether focusing on the hook/opening is sufficient or if full lyrics provide better vocabulary coverage.


In [ ]:
# extract chorus text for train and test sets
train_df['chorus_text'] = train_df['lyrics'].apply(extract_all_choruses).apply(lambda x: clean_for_classification(x, max_words=500))
test_df['chorus_text'] = test_df['lyrics'].apply(extract_all_choruses).apply(lambda x: clean_for_classification(x, max_words=500))

# audit chorus tag coverage
has_chorus = train_df['lyrics'].apply(lambda x: bool(re.search(r'\[Chorus', str(x), re.IGNORECASE)))
n_has_chorus = has_chorus.sum()
n_fallback = (~has_chorus).sum()
print(f"Songs with explicit [Chorus] tag : {n_has_chorus:,} ({n_has_chorus/len(train_df)*100:.1f}%)")
print(f"Songs using opening-stanza fallback: {n_fallback:,} ({n_fallback/len(train_df)*100:.1f}%)")

print(f"\nAverage words in full clean lyrics: {train_df['lyrics_clean_genre'].apply(lambda x: len(x.split())).mean():.1f}")
print(f"Average words in extracted chorus:   {train_df['chorus_text'].apply(lambda x: len(x.split())).mean():.1f}")


Songs with explicit [Chorus] tag : 38,013 (28.2%)
Songs using opening-stanza fallback: 96,954 (71.8%)

Average words in full clean lyrics: 154.7
Average words in extracted chorus:   22.5


## 5. Text Vectorization

We compare three representations:
1. **Bag-of-Words (CountVectorizer):** Standard term frequencies.
2. **TF-IDF (Unigrams + Bigrams):** Includes sublinear term frequency scaling (`1 + log(tf)`) and L2 normalization to account for differences in song length.
3. **Latent Semantic Analysis (TruncatedSVD):** Projects the TF-IDF matrix into 100 dense components.


In [ ]:
# 1. CountVectorizer baseline
count_vec = CountVectorizer(
    max_features=5000, 
    min_df=5, 
    max_df=0.85, 
    token_pattern=r"(?u)\b\w+\b"
)

# 2. TF-IDF with sublinear TF and bigrams
tfidf_vec = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.85,
    ngram_range=(1, 2),
    sublinear_tf=True,
    norm='l2',
    token_pattern=r"(?u)\b\w+\b"
)

print("Fitting TF-IDF Vectorizer on full cleaned lyrics...")
X_train_tfidf = tfidf_vec.fit_transform(train_df['lyrics_clean_genre'])
X_test_tfidf = tfidf_vec.transform(test_df['lyrics_clean_genre'])

# 3. TruncatedSVD (100 components)
svd = TruncatedSVD(n_components=100, random_state=42)
X_train_lsa = svd.fit_transform(X_train_tfidf)

print(f"TF-IDF Matrix shape:   {X_train_tfidf.shape} (Sparse)")
print(f"LSA Dense Matrix shape: {X_train_lsa.shape} (Dense, 100 components)")
print(f"Explained Variance by 100 LSA components: {svd.explained_variance_ratio_.sum() * 100:.2f}%")


Fitting TF-IDF Vectorizer on full cleaned lyrics...
TF-IDF Matrix shape:   (134967, 5000) (Sparse)
LSA Dense Matrix shape: (134967, 100) (Dense, 100 components)
Explained Variance by 100 LSA components: 13.76%


## 6. Feature Selection (Chi-Squared)

We use the Chi-squared test to inspect the top informative words associated with each genre.


In [ ]:
# encode target classes
le = LabelEncoder()
y_train = le.fit_transform(train_df['tag'])
classes = list(le.classes_)

# select top 3,000 features
chi2_selector = SelectKBest(score_func=chi2, k=3000)
X_train_selected = chi2_selector.fit_transform(X_train_tfidf, y_train)

# top 15 terms by chi2 score
feature_names = np.array(tfidf_vec.get_feature_names_out())
top_feature_indices = np.argsort(chi2_selector.scores_)[-15:][::-1]

top_terms_df = pd.DataFrame({
    'Term': feature_names[top_feature_indices],
    'Chi2 Score': chi2_selector.scores_[top_feature_indices]
})

plt.figure(figsize=(10, 5))
sns.barplot(data=top_terms_df, x='Chi2 Score', y='Term', palette='mako')
plt.title('Top 15 Most Informal/Discriminating Terms (Chi-Squared)', fontsize=14, weight='bold')
plt.xlabel('Chi-Squared Statistic')
plt.ylabel('Term')
plt.tight_layout()
plt.show()


NameError: name 'LabelEncoder' is not defined

The top terms reflect distinct genre vocabulary (e.g. slang terms in rap, distinct themes in rock and country).


## 7. Metadata Features

We also extract and normalize auxiliary metadata features (`has_features`, `word_count`, `year`, `log_views`) to see if adding tabular features to lyrics improves performance.


In [ ]:
def extract_metadata_features(df):
    meta = pd.DataFrame(index=df.index)
    
    # collaborations
    meta['has_features'] = df['features'].apply(
        lambda x: 1.0 if (x != '{}' and pd.notnull(x) and str(x).strip() != '') else 0.0
    )
    
    # word count
    meta['word_count'] = df['lyrics'].astype(str).apply(lambda x: len(x.split()))
    
    # release year (replace outliers with median)
    median_year = df['year'].median()
    meta['year'] = df['year'].apply(lambda y: y if (1900 <= y <= 2026) else median_year)
    
    # log views
    if 'views' in df.columns:
        meta['log_views'] = np.log1p(df['views'].clip(lower=0))
    else:
        meta['log_views'] = np.log1p(train_df['views'].median())
        
    return meta

train_meta = extract_metadata_features(train_df)
test_meta = extract_metadata_features(test_df)

scaler = StandardScaler()
X_train_meta_scaled = scaler.fit_transform(train_meta)
X_test_meta_scaled = scaler.transform(test_meta)

print(f"Extracted and scaled {X_train_meta_scaled.shape[1]} metadata features.")


Extracted and scaled 4 metadata features.


## 8. Comparing Preprocessing Configurations

We benchmark different configurations using a linear model (`SGDClassifier` with log loss) on an 80/20 stratified validation split.


In [ ]:
# encode labels
le = LabelEncoder()
y_train = le.fit_transform(train_df['tag'])

# 80/20 stratified split
train_idx, val_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=0.2,
    random_state=42,
    stratify=y_train
)
y_tr, y_val = y_train[train_idx], y_train[val_idx]
print(f"Train fold: {len(train_idx):,} songs | Validation fold: {len(val_idx):,} songs")

def run_config(config_name, X_train_fold, X_val_fold):
    clf = SGDClassifier(
        loss='log_loss', penalty='l2', alpha=1e-5,
        class_weight='balanced', random_state=42, max_iter=1000, tol=1e-3
    )
    clf.fit(X_train_fold, y_tr)
    preds = clf.predict(X_val_fold)
    macro_f1 = f1_score(y_val, preds, average='macro')
    return {'Configuration': config_name, 'Validation Macro-F1': round(macro_f1, 4)}

results = []
train_lyrics = train_df['lyrics_clean_genre']
train_chorus = train_df['chorus_text']

# Config A: CountVectorizer baseline
vec_a = CountVectorizer(max_features=5000, min_df=5, max_df=0.85, token_pattern=r"(?u)\b\w+\b")
X_a_tr = vec_a.fit_transform(train_lyrics.iloc[train_idx])
X_a_val = vec_a.transform(train_lyrics.iloc[val_idx])
results.append(run_config('Config A: Cleaned Text + CountVectorizer', X_a_tr, X_a_val))

# Config B: TF-IDF (unigrams)
vec_b = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.85, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")
X_b_tr = vec_b.fit_transform(train_lyrics.iloc[train_idx])
X_b_val = vec_b.transform(train_lyrics.iloc[val_idx])
results.append(run_config('Config B: TF-IDF (Unigrams)', X_b_tr, X_b_val))

# Config C: TF-IDF (unigrams + bigrams)
vec_c = TfidfVectorizer(
    max_features=5000, min_df=5, max_df=0.85,
    ngram_range=(1, 2), sublinear_tf=True, norm='l2', token_pattern=r"(?u)\b\w+\b"
)
X_c_tr = vec_c.fit_transform(train_lyrics.iloc[train_idx])
X_c_val = vec_c.transform(train_lyrics.iloc[val_idx])
results.append(run_config('Config C: TF-IDF (Unigrams + Bigrams)', X_c_tr, X_c_val))

# Config D: Chorus-only TF-IDF
vec_d = TfidfVectorizer(
    max_features=5000, min_df=5, max_df=0.85,
    ngram_range=(1, 2), sublinear_tf=True, norm='l2', token_pattern=r"(?u)\b\w+\b"
)
X_d_tr = vec_d.fit_transform(train_chorus.iloc[train_idx])
X_d_val = vec_d.transform(train_chorus.iloc[val_idx])
results.append(run_config('Config D: Chorus-Only TF-IDF', X_d_tr, X_d_val))

# Config E: LSA (100 components)
vec_e = TfidfVectorizer(
    max_features=5000, min_df=5, max_df=0.85,
    ngram_range=(1, 2), sublinear_tf=True, norm='l2', token_pattern=r"(?u)\b\w+\b"
)
X_e_sp_tr = vec_e.fit_transform(train_lyrics.iloc[train_idx])
X_e_sp_val = vec_e.transform(train_lyrics.iloc[val_idx])
svd_bench = TruncatedSVD(n_components=100, random_state=42)
X_e_tr = svd_bench.fit_transform(X_e_sp_tr)
X_e_val = svd_bench.transform(X_e_sp_val)
results.append(run_config('Config E: LSA (TF-IDF + SVD, 100 components)', X_e_tr, X_e_val))

# Config F: TF-IDF + metadata
train_meta = extract_metadata_features(train_df)
scaler_f = StandardScaler()
X_meta_tr = scaler_f.fit_transform(train_meta.iloc[train_idx])
X_meta_val = scaler_f.transform(train_meta.iloc[val_idx])
X_f_tr = hstack([X_c_tr, X_meta_tr]).tocsr()
X_f_val = hstack([X_c_val, X_meta_val]).tocsr()
results.append(run_config('Config F (Exploratory): TF-IDF + Metadata', X_f_tr, X_f_val))

# results
benchmark_df = pd.DataFrame(results).sort_values(by='Validation Macro-F1', ascending=False).reset_index(drop=True)
display(benchmark_df)


Train fold: 107,973 songs | Validation fold: 26,994 songs


d:\Python\envs\tm\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
d:\Python\envs\tm\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


,Configuration,Validation Macro-F1
0,Config C: TF-IDF (Unigrams + Bigrams),0.5027
1,Config B: TF-IDF (Unigrams),0.5005
2,Config F (Exploratory): TF-IDF + Metadata,0.4988
3,"Config E: LSA (TF-IDF + SVD, 100 components)",0.4688
4,Config D: Chorus-Only TF-IDF,0.3982
5,Config A: Cleaned Text + CountVectorizer,0.3969


### Summary of Results:
- **TF-IDF vs. CountVectorizer (Config B vs Config A):** TF-IDF provides a substantial improvement over raw word counts (~0.50 vs ~0.40 Macro-F1).
- **Unigrams + Bigrams with Sublinear TF (Config C):** Adding bigrams and sublinear TF scaling achieves the highest score (**0.5027 Macro-F1**), capturing multi-word genre markers while dampening high-frequency repetitive words.
- **Chorus-Only / Opening Section (Config D):** Scores lower (**0.3982 Macro-F1**). Because 71.8% of songs lack explicit chorus tags and fall back to opening lines, truncating to just the chorus/opening discards too much distinctive verse vocabulary.
- **Dimensionality Reduction via LSA (Config E):** Compressing TF-IDF into 100 dense SVD components (**0.4688 Macro-F1**) loses fine-grained keyword signals that distinguish genres.
- **Adding Metadata (Config F):** Adding scaled tabular metadata (`year`, `views`, `has_features`, `word_count`) to TF-IDF yields **0.4988 Macro-F1**, slightly underperforming text-only TF-IDF (0.5027). Since the brief focuses on lyrics-based classification, Config C is our primary representation.


## 9. Save Preprocessed Data
Saving the preprocessed text columns and fitted TF-IDF vectorizer for use in subsequent modeling.


In [ ]:
# save preprocessed data
os.makedirs('../tmp', exist_ok=True)

preproc_save_path = '../tmp/preprocessed_data.pickle'
save_dict = {
    'train_clean_genre': train_df['lyrics_clean_genre'],
    'train_clean_sentiment': train_df['lyrics_clean_sentiment'],
    'train_chorus': train_df['chorus_text'],
    'test_clean_genre': test_df['lyrics_clean_genre'],
    'test_clean_sentiment': test_df['lyrics_clean_sentiment'],
    'test_chorus': test_df['chorus_text'],
    'y_train': y_train,
    'classes': classes,
    'tfidf_vectorizer': tfidf_vec
}

with open(preproc_save_path, 'wb') as f:
    pickle.dump(save_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Preprocessed data saved to: {preproc_save_path}")
